## Pipeline Script 


Inputs: Group CEST and nmap data output from pyGluCEST as well as demographic data from _________. 
Outputs: Compiled dataframes with GluCEST and nmap data. Trimmed based on number of people with sufficient data.

    Trimmed subject-wise dfs: e.g., cestmat (outpath + 'trimmed_cestmat' + dataset + atlas + '.csv')
    Long form dfs: e.g., long_df (outpath + 'longform_grpdf' + dataset + '_' + atlas + '.csv')
         Also have version with standard nmap values
    Mean dfs: e.g., grouped_df (outpath + 'means_' + dataset + '_' + atlas + '.csv')


### Import Packages

In [19]:
import os
import glob
import numpy as np
import pandas as pd
#import network_fcon as fc
import scipy as sp
from scipy.stats import pearsonr
from scipy.stats import linregress
import seaborn as sns
import matplotlib.pyplot as plt
import re
from nilearn.datasets import fetch_atlas_schaefer_2018


### Define paths and variables

In [20]:
# Set variables
dataset = 'longglucest_outputmeasures2'
atlas = 'Schaefer2018_1000Parcels_17Networks'
atlas_name = 'schaefer'
if atlas_name == 'schaefer':
    scale='scale1000_17'
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17) # This is sklearn.utils._bunch.Bunch. Other altases in this format? *Ask Golia
    nnodes = len(schaefer['labels'])
    

receptors = ["5HT1a", "5HT1b", "5HT2a", "5HT4", "5HT6", "5HTT", "A4B2",
             "CB1", "D1", "D2", "DAT", "GABAa", "H3", "M1", "mGluR5",
             "MOR", "NET", "NMDA", "VAChT"]
features = ['abaPC1','CBF','CBV','CThick','Glucose','O2met','SAaxis']

nmaps = ["NMDA", "mGluR5", "GABA"]
maps = ["cest", "NMDA", "mGluR5", "GABA"]


# Set paths
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/" + dataset
outpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
outpath2 = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/" 

# Read in data
cestmat = pd.read_csv(inpath + "/all_subs_GluCEST_" + atlas + "_UNI.csv", sep=',')
cestmat.set_index('Subject', inplace = True)

# Load in standardized nmap data
receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_all_scale1000_17.csv", sep=',')
feature_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/feature_data_all_scale1000_17.csv", sep=',')


In [11]:
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
#print(grp_df)

## Trim Data

In [21]:
# ID parcels with < 20 voxels* 
for i, col in enumerate(cestmat.columns):
    if 'NZcount' in col:
        # Set mean col to nan
        mean_col = cestmat.columns[i - 1]
        sigma_col = cestmat.columns[i + 1]
        cestmat[mean_col] = np.where(cestmat[col] < 20, np.nan, cestmat[mean_col])
        cestmat[sigma_col] = np.where(cestmat[col] < 20, np.nan, cestmat[sigma_col])
        cestmat[col] = np.where(cestmat[col] < 20, np.nan, cestmat[col])       
columns = cestmat.columns[cestmat.notnull().sum() > len(cestmat)*.75]
print(cestmat.shape)

# Trim cest df based on column filter
cestmat= cestmat[columns]
print(cestmat.shape)

# Remove subjects missing >65% of remaining GluCEST parcels
sparse_subjs = cestmat[cestmat.isna().sum(axis=1) > cestmat.shape[1] * 0.65].index
cestmat = cestmat.drop(index=sparse_subjs)
print(cestmat.shape)

# Save trimmed dfs
cestmat.to_csv(outpath + '/trimmed_cestmat' + dataset + atlas + scale + '.csv', index=True)
cestmat.to_csv(outpath2 + '/trimmed_cestmat' + dataset + atlas + scale + '.csv', index=True)

(176, 830)
(176, 203)
(172, 203)


In [17]:
cestmat

,Unnamed: 0,group,17Networks_RH_VisCent_Striate_2 NZMean,17Networks_RH_VisCent_Striate_2 NZcount,17Networks_RH_VisCent_Striate_2 NZSigma,17Networks_RH_VisCent_ExStr_14 NZMean,17Networks_RH_VisCent_ExStr_14 NZcount,17Networks_RH_VisCent_ExStr_14 NZSigma,17Networks_RH_VisCent_ExStr_17 NZMean,17Networks_RH_VisCent_ExStr_17 NZcount,...,17Networks_RH_ContC_pCun_10 NZSigma,17Networks_RH_ContC_pCun_11 NZMean,17Networks_RH_ContC_pCun_11 NZcount,17Networks_RH_ContC_pCun_11 NZSigma,17Networks_RH_DefaultA_pCunPCC_5 NZMean,17Networks_RH_DefaultA_pCunPCC_5 NZcount,17Networks_RH_DefaultA_pCunPCC_5 NZSigma,17Networks_RH_DefaultA_pCunPCC_12 NZMean,17Networks_RH_DefaultA_pCunPCC_12 NZcount,17Networks_RH_DefaultA_pCunPCC_12 NZSigma
Subject,,,,,,,,,,,,,,,,,,,,,
100522_12003,0,TD/NC,NaN,NaN,NaN,11.568608,30.0,2.090549,11.563977,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100522_12371,1,TD/NC,NaN,NaN,NaN,NaN,NaN,NaN,12.899275,79.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100522_12783,2,TD/NC,NaN,NaN,NaN,NaN,NaN,NaN,12.170283,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102041_12037,3,PRO/CHR,6.896060,46.0,2.100414,7.324809,91.0,3.142994,6.183180,59.0,...,NaN,5.825535,23.0,2.046068,9.435372,28.0,3.355391,7.549210,23.0,2.533496
102041_12500,4,PRO/CHR,9.101267,56.0,1.292699,10.588638,104.0,3.798515,9.020026,59.0,...,2.031342,8.001007,28.0,1.791779,7.621892,29.0,1.634854,8.149293,23.0,1.433568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96902_11903,171,TD/NC,10.768049,32.0,1.255749,11.407769,27.0,1.396833,10.797512,88.0,...,1.151236,9.823335,62.0,1.501530,9.892211,38.0,1.099129,8.674353,37.0,1.250330
96902_12440,172,TD/NC,NaN,NaN,NaN,NaN,NaN,NaN,8.112674,57.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96902_12788,173,TD/NC,NaN,NaN,NaN,NaN,NaN,NaN,11.959593,62.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
feature_df

,Parcel,abaPC1,CBF,CBV,CThick,Glucose,O2met,SAaxis
0,17Networks_LH_VisCent_Striate_1,-1.017269,1.080034,-0.309541,-0.344899,-2.199974,-2.012666,-0.195458
1,17Networks_LH_VisCent_Striate_2,-1.528741,0.230634,-0.996347,-0.077911,-2.273773,-1.616657,-0.030261
2,17Networks_LH_VisCent_Striate_3,-1.335038,2.179686,2.491076,0.130418,-2.028452,0.989703,0.319988
3,17Networks_LH_VisCent_Striate_4,-2.391152,-1.332028,-1.267029,-0.397107,-2.150991,-1.594636,-0.168181
4,17Networks_LH_VisCent_ExStr_1,0.844841,-0.821664,-1.020337,-0.943399,-0.796223,-0.077485,-0.602312
...,...,...,...,...,...,...,...,...
995,17Networks_RH_TempPar_18,0.252809,-0.296236,0.183355,0.039980,0.995451,0.392090,0.046803
996,17Networks_RH_TempPar_19,-0.144144,0.320504,0.160905,0.726872,1.137494,-0.121955,-0.166595
997,17Networks_RH_TempPar_20,0.685681,-0.125140,0.285438,0.288808,0.805814,0.870236,-0.191560
998,17Networks_RH_TempPar_21,0.306854,-0.205778,0.195602,0.987100,1.273230,0.505388,-0.241407


In [36]:
receptor_df

,Parcel,5HT1a,5HT1b,5HT2a,5HT4,5HT6,5HTT,A4B2,CB1,D1,D2,DAT,GABAa,H3,M1,mGluR5,MOR,NET,NMDA,VAChT
Parcel,,,,,,,,,,,,,,,,,,,,
17Networks_LH_VisCent_Striate_1,17Networks_LH_VisCent_Striate_1,18.948129,0.041845,48.899766,1.214659,0.076800,4.452689,5.702382,1.120613,10.222895,-1.793413,27.112566,801.482003,12.104925,2.123639,-1.676590,0.000000,-0.165843,6.506956,-0.284921
17Networks_LH_VisCent_Striate_2,17Networks_LH_VisCent_Striate_2,14.188564,-0.686076,51.189834,1.184958,0.136049,4.699549,6.015368,1.171355,9.386907,-1.725353,25.816654,876.656020,13.585886,2.649931,-1.582004,0.000000,-0.146195,5.825221,-0.280327
17Networks_LH_VisCent_Striate_3,17Networks_LH_VisCent_Striate_3,15.991089,2.412334,52.976763,1.283687,0.525998,5.222937,8.935845,1.115665,13.823058,-1.079382,27.518347,973.196315,16.292319,4.259038,-0.612121,0.000000,0.034422,7.150124,-0.090520
17Networks_LH_VisCent_Striate_4,17Networks_LH_VisCent_Striate_4,6.163279,-1.283465,48.451955,1.234372,0.139344,4.493451,6.665540,1.171664,4.613154,-1.734788,24.172900,720.693036,13.789880,3.392128,-2.019832,0.000000,-0.090363,5.112683,-0.333713
17Networks_LH_VisCent_ExStr_1,17Networks_LH_VisCent_ExStr_1,36.276247,-0.970097,43.766169,1.779596,0.366331,3.844715,10.412034,1.351787,16.448932,0.268522,30.359035,821.139673,16.388692,2.804235,-0.341650,19.869329,0.012567,7.171719,0.062423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17Networks_RH_TempPar_18,17Networks_RH_TempPar_18,30.767017,0.062079,52.201041,2.500652,0.436586,4.814709,10.125429,1.191691,19.034882,0.148707,30.644053,782.871657,12.937686,3.255362,0.748027,21.933858,0.055272,7.154420,0.034462
17Networks_RH_TempPar_19,17Networks_RH_TempPar_19,27.073119,0.201264,58.092809,2.557817,0.359678,4.495821,8.995112,1.342542,12.808795,0.217057,26.675196,729.726225,18.514514,4.238721,0.810461,19.715440,0.074454,6.208405,-0.014531
17Networks_RH_TempPar_20,17Networks_RH_TempPar_20,34.795170,0.181614,54.335341,2.424333,0.508124,4.458515,10.177820,1.266869,17.551809,0.635338,30.020101,762.064750,15.281439,4.008876,0.746511,22.437446,0.035453,7.109887,0.020519


# Add neuromaps data standard nmaps

In [32]:
# Import and add parcel labels to standard receptor_df
"""
schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
labels = schaefer.labels
labels = [label.decode('utf-8') for label in labels]
receptor_df.index = labels
receptor_df.index.name = 'Parcel'
feature_df.index = labels
feature_df.index.name = 'Parcel'
"""

### Make classic grp_df

In [9]:
grp_df_std

,Unnamed: 0,group,17Networks_RH_VisCent_Striate_2 NZMean,17Networks_RH_VisCent_Striate_2 NZcount,17Networks_RH_VisCent_Striate_2 NZSigma,17Networks_RH_VisCent_ExStr_14 NZMean,17Networks_RH_VisCent_ExStr_14 NZcount,17Networks_RH_VisCent_ExStr_14 NZSigma,17Networks_RH_VisCent_ExStr_17 NZMean,17Networks_RH_VisCent_ExStr_17 NZcount,...,17Networks_RH_ContC_pCun_10 NZSigma,17Networks_RH_ContC_pCun_11 NZMean,17Networks_RH_ContC_pCun_11 NZcount,17Networks_RH_ContC_pCun_11 NZSigma,17Networks_RH_DefaultA_pCunPCC_5 NZMean,17Networks_RH_DefaultA_pCunPCC_5 NZcount,17Networks_RH_DefaultA_pCunPCC_5 NZSigma,17Networks_RH_DefaultA_pCunPCC_12 NZMean,17Networks_RH_DefaultA_pCunPCC_12 NZcount,17Networks_RH_DefaultA_pCunPCC_12 NZSigma
100522_12003,0.0,TD/NC,NaN,NaN,NaN,11.568608,30.0,2.090549,11.563977,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100522_12371,1.0,TD/NC,NaN,NaN,NaN,NaN,NaN,NaN,12.899275,79.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100522_12783,2.0,TD/NC,NaN,NaN,NaN,NaN,NaN,NaN,12.170283,92.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102041_12037,3.0,PRO/CHR,6.896060,46.0,2.100414,7.324809,91.0,3.142994,6.183180,59.0,...,NaN,5.825535,23.0,2.046068,9.435372,28.0,3.355391,7.549210,23.0,2.533496
102041_12500,4.0,PRO/CHR,9.101267,56.0,1.292699,10.588638,104.0,3.798515,9.020026,59.0,...,2.031342,8.001007,28.0,1.791779,7.621892,29.0,1.634854,8.149293,23.0,1.433568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
## Edited to loop through all receptors:
"""
receptor_types = receptor_df.columns.tolist()
filtered_receptor_dfs = []

# Loop through each receptor type in the original receptor_df
for receptor in receptor_types:
    # Transpose the receptor map
    receptor_transposed = receptor_df[[receptor]].T

    # Keep only parcels contained in cestmat
    cestmat_regions = [col.replace(' NZMean', '') for col in cestmat.columns if ' NZMean' in col]
    receptor_filtered = receptor_transposed[[col for col in receptor_transposed.columns if col in cestmat_regions]]

    # Rename the columns to include the receptor type prefix
    receptor_filtered.columns = [f"{receptor}_{col}" for col in receptor_filtered.columns]

    # Repeat the filtered data to match the length of cestmat
    receptor_repeated = pd.concat([receptor_filtered] * len(cestmat), ignore_index=True)

    # Append the processed receptor data to the list
    filtered_receptor_dfs.append(receptor_repeated)


# Concatenate all receptor data with cestmat
grp_df_std = pd.concat([cestmat] + filtered_receptor_dfs, axis=1)

# Save the final dataframe
grp_df_std.to_csv(outpath + '/grp_df_std_all_receptors' + dataset + atlas + '.csv', index=True)
grp_df_std.to_csv(outpath2 + '/grp_df_std_all_receptors' + dataset + atlas + '.csv', index=True)


print(grp_df_std.shape)
"""

(344, 203)


### Make Longform df

In [25]:
cestlong

,Subject,Parcel,GluCEST
0,0,17Networks_RH_VisCent_Striate_2 NZMean,NaN
1,1,17Networks_RH_VisCent_Striate_2 NZMean,NaN
2,2,17Networks_RH_VisCent_Striate_2 NZMean,NaN
3,3,17Networks_RH_VisCent_Striate_2 NZMean,6.896060
4,4,17Networks_RH_VisCent_Striate_2 NZMean,9.101267
...,...,...,...
23043,339,17Networks_RH_DefaultA_pCunPCC_12 NZMean,NaN
23044,340,17Networks_RH_DefaultA_pCunPCC_12 NZMean,NaN
23045,341,17Networks_RH_DefaultA_pCunPCC_12 NZMean,NaN
23046,342,17Networks_RH_DefaultA_pCunPCC_12 NZMean,NaN


In [22]:
# Make longform df using standardized nmaps values

#grp_df_std.rename_axis('Subject', inplace=True)
#grp_df_std.reset_index(inplace=True)
# Get list of parcel names
parcels = cestmat.filter(like="NZMean").columns.tolist()

# Melt cestmat to get Glu data in long format
cestlong = grp_df_std.reset_index().melt(id_vars='Subject', value_vars=parcels, 
                                      var_name='Parcel', value_name='GluCEST')

cestlong



"""
# Step 2: Prepare receptor data without subject information
# Loop through receptors to align receptor data with parcels
long_dfs = []
for receptor in receptor_types:  # Loop through receptor names
    receptor_long = grp_df_std[[col for col in grp_df_std.columns if col.startswith(receptor)]]
    
    # Extract the parcel names
    receptor_long.columns = [col.replace(f'{receptor}_', '') for col in receptor_long.columns]
    
    # Transpose receptor data for easier alignment
    receptor_long = receptor_long.T.reset_index()
    receptor_long.columns = ['Parcel', receptor]  # Rename columns for clarity
    
    long_dfs.append(receptor_long)

# Step 3: Merge receptor data into cestlong
for receptor_long in long_dfs:
    cestlong = cestlong.merge(receptor_long, on='Parcel', how='left')

# Final long-form dataframe
print(cestlong)












longdf_cest = long_df[["Subject","Parcel","GluCEST","group","hstatus"]]
longdf_cest["Parcel"] = longdf_cest["Parcel"].str.replace(' NZMean', '', regex=False)

# Convert receptor_df from wide to long format for merging
nmda_long = NMDAmat.reset_index().melt(id_vars='Parcel', var_name='Receptor1', value_name='NMDA_standard')
gaba_long = GABAmat.reset_index().melt(id_vars='Parcel', var_name='Receptor2', value_name='GABA_standard')
mglur5_long = mGluR5mat.reset_index().melt(id_vars='Parcel', var_name='Receptor3', value_name='mGluR5_standard')

# Combine dfs
long_df_std = pd.merge(longdf_cest, nmda_long, on='Parcel', how='left')
long_df_std = pd.merge(long_df_std, gaba_long, on='Parcel', how='left')
long_df_std = pd.merge(long_df_std, mglur5_long, on='Parcel', how='left')
# Remove unnecessary columns
long_df_std = long_df_std.drop(columns=long_df_std.filter(like="Receptor").columns)
#print(long_df_std)

# Save the longform dataframe to a CSV
long_df_std.to_csv(outpath + '/longform_grpdf_std_' + dataset + '_' + atlas + '.csv', index=False)
"""

ValueError: cannot insert Subject, already exists

### Make mean df by diagnosis

In [78]:
# Make mean group dfs by diagnosis
# Standard nmap data
grouped_std = long_df_std.groupby(['Parcel', 'hstatus']).agg(
    GluCEST_avg=('GluCEST', 'mean'),
    NMDA_avg=('NMDA_standard', 'mean'),
    mGluR5_avg=('mGluR5_standard', 'mean'),
    GABA_avg=('GABA_standard', 'mean')
).reset_index()
#print(grouped_std)
grouped_std.to_csv(outpath + '/means_std_' + dataset + '_' + atlas + '.csv', index=False)

NameError: name 'long_df_std' is not defined

### Data Imputation

In [213]:
# Now, for the long_dfs, impute data based on average across participants for that parcel
# Subject-wise data
merged_df = pd.merge(long_df, grouped_df2[['Parcel', 'hstatus', 'GluCEST_avg']], on=['Parcel', 'hstatus'], how='left')
merged_df['GluCEST'] = merged_df['GluCEST'].fillna(merged_df['GluCEST_avg'])
#print(merged_df)
imputed_df = merged_df.drop(columns=['GluCEST_avg'])

# Standard data
# First, rename parcels
long_df_std["Parcel"] = long_df_std["Parcel"].str.replace(' NZMean', '', regex=False)
#print(long_df_std)
merged_df_std = pd.merge(long_df_std, grouped_df[['Parcel', 'hstatus', 'GluCEST_avg']], on=['Parcel', 'hstatus'], how='left')
merged_df_std['GluCEST'] = merged_df_std ['GluCEST'].fillna(merged_df_std ['GluCEST_avg'])
#print(merged_df_std)
imputed_df_std = merged_df_std.drop(columns=['GluCEST_avg'])

imputed_df.to_csv(outpath + '/imputed_long_df_' + dataset + '_' + atlas + '.csv', index=False)
imputed_df_std.to_csv(outpath + '/imputed_long_df_standardnmaps_' + dataset + '_' + atlas + '.csv', index=False)

### Normalize GluCEST values

In [99]:
# Step 1: Select columns that contain 'NZMean'
nzmean_columns = [col for col in cestmat.columns if 'NZMean' in col]
print(nzmean_columns)

# Step 2: Calculate mean and std deviation for each subject (row-wise) across selected columns
cestmat['Subject_Avg_NZMean'] = cestmat[nzmean_columns].mean(axis=1)
cestmat['Subject_Std_NZMean'] = cestmat[nzmean_columns].std(axis=1)
print(cestmat)

# Step 3: Calculate z-scores for all selected columns at once and store them in a new dataframe
zscore_df = (cestmat[nzmean_columns].sub(cestmat['Subject_Avg_NZMean'], axis=0)
             .div(cestmat['Subject_Std_NZMean'], axis=0))

# Rename z-score columns
zscore_df.columns = [col + '_Zscore' for col in zscore_df.columns]
print(zscore_df.size)
print(zscore_df)
# Step 4: Concatenate the z-scores dataframe to the original cestmat dataframe
zcestmat = pd.concat([cestmat['group'], zscore_df], axis=1)

print(zcestmat)


['17Networks_RH_VisCent_Striate_2 NZMean', '17Networks_RH_VisCent_ExStr_14 NZMean', '17Networks_RH_VisCent_ExStr_17 NZMean', '17Networks_RH_VisPeri_StriCal_3 NZMean', '17Networks_RH_VisPeri_StriCal_5 NZMean', '17Networks_RH_VisPeri_ExStrInf_7 NZMean', '17Networks_RH_VisPeri_ExStrSup_3 NZMean', '17Networks_RH_SomMotA_6 NZMean', '17Networks_RH_SomMotA_12 NZMean', '17Networks_RH_SomMotA_22 NZMean', '17Networks_RH_SalVentAttnA_ParMed_8 NZMean', '17Networks_RH_SalVentAttnA_FrMed_1 NZMean', '17Networks_RH_SalVentAttnA_FrMed_2 NZMean', '17Networks_RH_SalVentAttnA_FrMed_3 NZMean', '17Networks_RH_SalVentAttnA_FrMed_4 NZMean', '17Networks_RH_SalVentAttnA_FrMed_5 NZMean', '17Networks_RH_SalVentAttnA_FrMed_6 NZMean', '17Networks_RH_SalVentAttnA_FrMed_7 NZMean', '17Networks_RH_SalVentAttnA_FrMed_8 NZMean', '17Networks_RH_SalVentAttnB_PFCmp_1 NZMean', '17Networks_RH_SalVentAttnB_PFCmp_2 NZMean', '17Networks_RH_SalVentAttnB_PFCmp_5 NZMean', '17Networks_RH_SalVentAttnB_Cinga_1 NZMean', '17Networks_RH_

In [ ]:
cestmat2=cestmat.copy()
NMDAmat2=NMDAmat.copy()
GABAmat2=GABAmat.copy()
mGluR5mat2=mGluR5mat.copy()
cestmat2.columns = [f"GluCEST_{col}" if "NZ" in col else col for col in cestmat2.columns]
NMDAmat2.columns = [f"NMDA_{col}" if "NZ" in col else col for col in NMDAmat2.columns]
GABAmat2.columns = [f"GABA_{col}" if "NZ" in col else col for col in GABAmat2.columns]
mGluR5mat2.columns = [f"mGluR5_{col}" if "NZ" in col else col for col in mGluR5mat2.columns]

# Align dataframes by "Subject" index and concatenate along columns
grp_df = cestmat2.join(NMDAmat2.filter(like='NZ'), how='left')
grp_df = grp_df.join(GABAmat2.filter(like='NZ'), how='left')
grp_df = grp_df.join(mGluR5mat2.filter(like='NZ'), how='left')

# Save grp_df
grp_df.to_csv(outpath + '/grp_df_' + dataset + atlas + '.csv', index=True)

In [ ]:
# Start with Loop through subjects. 
# Find path to 

I have parcel-wise glucest values and I want to normalize based on that subject's average glucest value.
Loop through rows of df
subj= the index of that row. 
Get average glucest value from path that includes subj in the path name

# Transpose receptor maps.
nmda = NMDAmat.T
gaba = GABAmat.T
mglur5 = mGluR5mat.T

# Keep only parcels contained in cestmat
cestmat_regions = [col.replace(' NZMean', '') for col in cestmat.columns if ' NZMean' in col]
nmda_filtered = nmda[[col for col in nmda.columns if col in cestmat_regions]]
gaba_filtered = gaba[[col for col in gaba.columns if col in cestmat_regions]]
mglur5_filtered = mglur5[[col for col in mglur5.columns if col in cestmat_regions]]

# Filtered columns
nmda_filtered.columns = [f"NMDA_{col}" for col in nmda_filtered.columns]
gaba_filtered.columns = [f"GABA_{col}" for col in gaba_filtered.columns]
mglur5_filtered.columns = [f"mGluR5_{col}" for col in mglur5_filtered.columns]

# Repeat values for length of cestmat
nmda_repeated = pd.concat([nmda_filtered] * len(cestmat), ignore_index=True)
gaba_repeated = pd.concat([gaba_filtered] * len(cestmat), ignore_index=True)
mglur5_repeated = pd.concat([mglur5_filtered] * len(cestmat), ignore_index=True)

# Concatenate
grp_df_std = pd.concat([cestmat2, nmda_repeated, gaba_repeated, mglur5_repeated], axis=1)

# Save grp_df_std
grp_df_std.to_csv(outpath + '/grp_df_std' + dataset + atlas + '.csv', index=True)
print(grp_df_std)